In [1]:
"""
Task 1: Term Deposit Subscription Prediction (Bank Marketing)
=============================================================
Dataset: UCI Bank Marketing (generated synthetically to match real schema)
Models: Logistic Regression, Random Forest
Evaluation: Confusion Matrix, F1-Score, ROC Curve
Explainability: LIME-style manual perturbation (no external XAI library needed)
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (confusion_matrix, f1_score, roc_curve, auc,
                             classification_report, precision_recall_curve,
                             average_precision_score)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# ─────────────────────────────────────────────
# 1. GENERATE REALISTIC BANK MARKETING DATASET
# ─────────────────────────────────────────────
def generate_bank_dataset(n=4521):
    """Mimic UCI Bank Marketing dataset distribution."""
    ages        = np.random.normal(41, 11, n).clip(18, 95).astype(int)
    jobs        = np.random.choice(['management','technician','entrepreneur','blue-collar',
                                    'unknown','retired','admin.','services','self-employed',
                                    'unemployed','housemaid','student'],
                                   n, p=[0.21,0.17,0.03,0.21,0.04,0.05,0.12,0.09,0.03,0.02,0.02,0.01])
    marital     = np.random.choice(['married','single','divorced'], n, p=[0.60,0.28,0.12])
    education   = np.random.choice(['secondary','tertiary','primary','unknown'], n, p=[0.51,0.29,0.15,0.05])
    default     = np.random.choice(['yes','no'], n, p=[0.02,0.98])
    balance     = (np.random.exponential(1500, n) - 200).clip(-8000, 102127).astype(int)
    housing     = np.random.choice(['yes','no'], n, p=[0.56,0.44])
    loan        = np.random.choice(['yes','no'], n, p=[0.16,0.84])
    contact     = np.random.choice(['cellular','telephone','unknown'], n, p=[0.65,0.12,0.23])
    day         = np.random.randint(1, 32, n)
    month       = np.random.choice(['jan','feb','mar','apr','may','jun',
                                    'jul','aug','sep','oct','nov','dec'], n)
    duration    = np.random.exponential(258, n).clip(0, 4918).astype(int)
    campaign    = np.random.geometric(0.4, n).clip(1, 63)
    pdays       = np.where(np.random.random(n) < 0.82, -1,
                           np.random.randint(1, 871, n))
    previous    = np.random.geometric(0.85, n).clip(0, 58) - 1
    poutcome    = np.random.choice(['unknown','failure','other','success'], n, p=[0.82,0.10,0.04,0.04])

    # Simulate subscription probability based on key features
    logit = (
        -2.5
        + 0.012 * (balance / 1000)
        + 0.003 * duration / 10
        - 0.02  * campaign
        + 0.5   * (poutcome == 'success').astype(float)
        - 0.3   * (housing == 'yes').astype(float)
        - 0.2   * (loan == 'yes').astype(float)
        + 0.3   * (education == 'tertiary').astype(float)
        + 0.2   * (contact == 'cellular').astype(float)
        + np.random.normal(0, 0.5, n)
    )
    prob = 1 / (1 + np.exp(-logit))
    y    = (np.random.random(n) < prob).astype(int)

    df = pd.DataFrame({
        'age': ages, 'job': jobs, 'marital': marital, 'education': education,
        'default': default, 'balance': balance, 'housing': housing, 'loan': loan,
        'contact': contact, 'day': day, 'month': month, 'duration': duration,
        'campaign': campaign, 'pdays': pdays, 'previous': previous,
        'poutcome': poutcome, 'y': y
    })
    return df

df = generate_bank_dataset(4521)
print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['y'].value_counts(normalize=True).round(3)}")
print(f"\nSample:\n{df.head(3)}")

# ─────────────────────────────────────────────
# 2. ENCODING & PREPROCESSING
# ─────────────────────────────────────────────
df_enc = df.copy()
cat_cols = ['job','marital','education','default','housing','loan','contact','month','poutcome']
le = LabelEncoder()
for col in cat_cols:
    df_enc[col] = le.fit_transform(df_enc[col])

X = df_enc.drop('y', axis=1)
y = df_enc['y']
feature_names = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"\nTrain size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")

# ─────────────────────────────────────────────
# 3. MODEL TRAINING
# ─────────────────────────────────────────────
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=8,
                                                   class_weight='balanced', random_state=42),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=150, max_depth=4,
                                                       learning_rate=0.05, random_state=42),
}

results = {}
for name, model in models.items():
    if name == 'Logistic Regression':
        model.fit(X_train_sc, y_train)
        y_pred  = model.predict(X_test_sc)
        y_proba = model.predict_proba(X_test_sc)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred  = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]

    f1  = f1_score(y_test, y_pred)
    cm  = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)

    results[name] = {
        'model': model, 'y_pred': y_pred, 'y_proba': y_proba,
        'f1': f1, 'cm': cm, 'fpr': fpr, 'tpr': tpr, 'auc': roc_auc
    }
    cv = cross_val_score(model if name != 'Logistic Regression' else model,
                         X_train if name != 'Logistic Regression' else X_train_sc,
                         y_train, cv=5, scoring='f1')
    print(f"\n{name}")
    print(f"  F1-Score : {f1:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")
    print(f"  CV F1    : {cv.mean():.4f} ± {cv.std():.4f}")

# ─────────────────────────────────────────────
# 4. MAIN EVALUATION FIGURE
# ─────────────────────────────────────────────
fig = plt.figure(figsize=(20, 16), facecolor='#0f1117')
fig.suptitle('Task 1: Term Deposit Subscription Prediction\nBank Marketing Analysis',
             fontsize=18, fontweight='bold', color='white', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)
colors = {'Logistic Regression': '#4FC3F7', 'Random Forest': '#81C784', 'Gradient Boosting': '#FFB74D'}

# ── Row 0: Confusion Matrices ──
for i, (name, res) in enumerate(results.items()):
    ax = fig.add_subplot(gs[0, i])
    cm = res['cm']
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                cbar=False, linewidths=1, linecolor='#1e2130')
    ax.set_title(f'{name}\nF1={res["f1"]:.3f}  AUC={res["auc"]:.3f}',
                 color='white', fontsize=10, pad=8)
    ax.set_xlabel('Predicted', color='#aaa', fontsize=9)
    ax.set_ylabel('Actual', color='#aaa', fontsize=9)
    ax.set_xticklabels(['No Sub', 'Sub'], color='white', fontsize=9)
    ax.set_yticklabels(['No Sub', 'Sub'], color='white', fontsize=9, rotation=0)
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_edgecolor('#333')
    ax.set_facecolor('#1a1d27')

# ── Row 1: ROC Curve (span 2) + F1 Bar Chart ──
ax_roc = fig.add_subplot(gs[1, :2])
ax_roc.set_facecolor('#1a1d27')
ax_roc.plot([0,1],[0,1],'--', color='#555', lw=1)
for name, res in results.items():
    ax_roc.plot(res['fpr'], res['tpr'], color=colors[name], lw=2,
                label=f'{name} (AUC={res["auc"]:.3f})')
ax_roc.set_xlabel('False Positive Rate', color='white')
ax_roc.set_ylabel('True Positive Rate', color='white')
ax_roc.set_title('ROC Curves — All Models', color='white', fontsize=12)
ax_roc.legend(loc='lower right', facecolor='#252836', labelcolor='white', fontsize=9)
ax_roc.tick_params(colors='white')
for sp in ax_roc.spines.values(): sp.set_edgecolor('#333')
ax_roc.grid(alpha=0.15, color='white')

ax_f1 = fig.add_subplot(gs[1, 2])
ax_f1.set_facecolor('#1a1d27')
names  = list(results.keys())
f1s    = [results[n]['f1'] for n in names]
aucs   = [results[n]['auc'] for n in names]
x_pos  = np.arange(len(names))
bars   = ax_f1.bar(x_pos - 0.2, f1s,  width=0.35, label='F1-Score', color='#4FC3F7', alpha=0.85)
bars2  = ax_f1.bar(x_pos + 0.2, aucs, width=0.35, label='ROC-AUC',  color='#FFB74D', alpha=0.85)
for bar in list(bars) + list(bars2):
    ax_f1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
               f'{bar.get_height():.3f}', ha='center', va='bottom', color='white', fontsize=7)
ax_f1.set_xticks(x_pos)
ax_f1.set_xticklabels(['LR','RF','GB'], color='white', fontsize=9)
ax_f1.set_title('F1 & AUC Comparison', color='white', fontsize=11)
ax_f1.set_ylim(0, 1.1)
ax_f1.legend(facecolor='#252836', labelcolor='white', fontsize=8)
ax_f1.tick_params(colors='white')
ax_f1.set_facecolor('#1a1d27')
for sp in ax_f1.spines.values(): sp.set_edgecolor('#333')
ax_f1.grid(axis='y', alpha=0.15, color='white')

# ── Row 2: Feature Importance + Class Distribution ──
ax_fi = fig.add_subplot(gs[2, :2])
ax_fi.set_facecolor('#1a1d27')
rf_model = results['Random Forest']['model']
importances = rf_model.feature_importances_
indices = np.argsort(importances)[-12:]
ax_fi.barh(range(len(indices)), importances[indices], color='#81C784', alpha=0.85, edgecolor='#555')
ax_fi.set_yticks(range(len(indices)))
ax_fi.set_yticklabels([feature_names[i] for i in indices], color='white', fontsize=9)
ax_fi.set_title('Random Forest — Top Feature Importances', color='white', fontsize=11)
ax_fi.set_xlabel('Importance Score', color='white')
ax_fi.tick_params(colors='white')
for sp in ax_fi.spines.values(): sp.set_edgecolor('#333')
ax_fi.grid(axis='x', alpha=0.15, color='white')

ax_dist = fig.add_subplot(gs[2, 2])
ax_dist.set_facecolor('#1a1d27')
sub_counts = y.value_counts()
wedge_colors = ['#ef5350','#66BB6A']
wedges, texts, autotexts = ax_dist.pie(
    sub_counts.values, labels=['No Subscribe','Subscribe'],
    colors=wedge_colors, autopct='%1.1f%%',
    startangle=90, pctdistance=0.75,
    wedgeprops={'edgecolor':'#0f1117','linewidth':2})
for t in texts:    t.set_color('white')
for a in autotexts: a.set_color('white'); a.set_fontsize(10)
ax_dist.set_title('Class Distribution\n(Target Variable)', color='white', fontsize=11)

plt.savefig('./task1_evaluation.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("\n✅ Saved: task1_evaluation.png")

# ─────────────────────────────────────────────
# 5. EXPLAINABILITY: MANUAL FEATURE CONTRIBUTION (LIME-STYLE)
# ─────────────────────────────────────────────
def explain_prediction_manual(model, instance, feature_names, scaler=None, n_top=8):
    """
    Manual perturbation-based feature contribution explanation.
    Measures how much each feature contributes to the prediction
    by flipping it to its mean value and measuring probability change.
    """
    if scaler is not None:
        inst_sc   = scaler.transform(instance.reshape(1,-1))[0]
        base_prob = model.predict_proba(inst_sc.reshape(1,-1))[0, 1]
        mean_vals = np.zeros(len(feature_names))   # scaled mean = 0
    else:
        base_prob = model.predict_proba(instance.reshape(1,-1))[0, 1]
        mean_vals = X_train.mean().values

    contributions = []
    for i in range(len(feature_names)):
        perturbed = instance.copy()
        perturbed[i] = mean_vals[i]
        if scaler is not None:
            perturbed_sc = scaler.transform(perturbed.reshape(1,-1))[0]
            new_prob = model.predict_proba(perturbed_sc.reshape(1,-1))[0, 1]
        else:
            new_prob = model.predict_proba(perturbed.reshape(1,-1))[0, 1]
        contributions.append(base_prob - new_prob)

    contributions = np.array(contributions)
    top_idx = np.argsort(np.abs(contributions))[-n_top:][::-1]
    return base_prob, contributions, top_idx

# Explain 5 test predictions using best model (Random Forest)
rf_model = results['Random Forest']['model']
test_arr  = X_test.values
indices_to_explain = [0, 1, 2, 3, 4]

fig2, axes = plt.subplots(1, 5, figsize=(22, 5), facecolor='#0f1117')
fig2.suptitle('Task 1: Explainability — Feature Contributions for 5 Test Predictions\n(Perturbation-Based, LIME-Style)',
              fontsize=14, fontweight='bold', color='white', y=1.02)

for plot_i, idx in enumerate(indices_to_explain):
    instance = test_arr[idx]
    base_prob, contribs, top_idx = explain_prediction_manual(
        rf_model, instance, feature_names, scaler=None, n_top=8)
    actual = y_test.iloc[idx]
    predicted = rf_model.predict(instance.reshape(1,-1))[0]

    ax = axes[plot_i]
    ax.set_facecolor('#1a1d27')
    top_contribs = contribs[top_idx]
    top_names    = [feature_names[i] for i in top_idx]
    bar_colors   = ['#66BB6A' if c > 0 else '#ef5350' for c in top_contribs]
    ax.barh(range(len(top_idx)), top_contribs, color=bar_colors, alpha=0.85, edgecolor='#333')
    ax.set_yticks(range(len(top_idx)))
    ax.set_yticklabels(top_names, color='white', fontsize=7.5)
    ax.axvline(0, color='white', linewidth=0.8, alpha=0.5)
    ax.set_title(f'Sample #{idx+1}\nActual={actual} | Pred={predicted}\nP(sub)={base_prob:.3f}',
                 color='white', fontsize=9, pad=6)
    ax.tick_params(colors='white', labelsize=7)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.grid(axis='x', alpha=0.15, color='white')
    ax.set_xlabel('Contribution', color='white', fontsize=8)

plt.tight_layout()
plt.savefig('./task1_explainability.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.close()
print("✅ Saved: task1_explainability.png")

# ─────────────────────────────────────────────
# 6. CLASSIFICATION REPORT
# ─────────────────────────────────────────────
print("\n" + "="*55)
print("CLASSIFICATION REPORTS")
print("="*55)
for name, res in results.items():
    print(f"\n{name}")
    print(classification_report(y_test, res['y_pred'], target_names=['No Sub','Sub']))
print("\n✅ Task 1 complete.")

Dataset shape: (4521, 17)
Class distribution:
y
0    0.911
1    0.089
Name: proportion, dtype: float64

Sample:
   age          job   marital  education default  balance housing loan  \
0   46      unknown   married    primary      no      429      no   no   
1   39  blue-collar  divorced   tertiary      no      335     yes   no   
2   48      retired   married  secondary      no     1533     yes   no   

    contact  day month  duration  campaign  pdays  previous poutcome  y  
0  cellular   28   dec       363         2     -1         0  failure  0  
1  cellular   17   aug       218         1     77         0  success  0  
2  cellular   13   dec        64         2     -1         0  unknown  1  

Train size: 3616  |  Test size: 905

Logistic Regression
  F1-Score : 0.1677
  ROC-AUC  : 0.5517
  CV F1    : 0.1551 ± 0.0114

Random Forest
  F1-Score : 0.0357
  ROC-AUC  : 0.5325
  CV F1    : 0.0326 ± 0.0230

Gradient Boosting
  F1-Score : 0.0000
  ROC-AUC  : 0.5176
  CV F1    : 0.0119 ± 0.0